In [ ]:
import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import precision_recall_curve, auc
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from tqdm import tqdm

In [ ]:
# Test kernel
print("Kernel is working!")
import psutil
print(f"Memory usage: {psutil.virtual_memory().percent}%")

In [ ]:
# --- 1. Dataset Version Used for Training ---
# Confirming dataset source and initial load
print("Loading dataset...")
df = pd.read_parquet("../data/processed/dataset_192H_60_forecast.parquet")
print(f"Dataset loaded with {len(df)} rows and {len(df.columns)} columns.")

### Initial EDA - First Look at the Dataset

This section adds a quick exploratory pass right after loading the data:

- shape and sample preview
- missing-value overview
- target distribution
- key numeric feature distributions
- hourly pattern of delays

In [ ]:
# 1) EDA setup + quick structure check
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
display(df.head(3))
display(df.describe(include="all").T.head(12))

In [ ]:
# 2) Missing values overview
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_top = missing_pct[missing_pct > 0].head(15)

print("Columns with missing values (top 15):")
print(missing_top if not missing_top.empty else "No missing values detected.")

if not missing_top.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(x=missing_top.values, y=missing_top.index, ax=ax)
    ax.set_title("Top Missing-Value Percentages")
    ax.set_xlabel("Missing %")
    ax.set_ylabel("Column")
    plt.tight_layout()
    plt.show()

In [ ]:
# 3) Target and categorical quick checks
if "late_3min" in df.columns:
    target_share = df["late_3min"].value_counts(normalize=True).sort_index()
    print("late_3min distribution (share):")
    print(target_share)

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.countplot(data=df, x="late_3min", ax=ax)
    ax.set_title("Target Distribution: late_3min")
    ax.set_xlabel("late_3min")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("Column 'late_3min' not found.")

if "line_id" in df.columns:
    top_lines = df["line_id"].value_counts().head(10)
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(x=top_lines.index.astype(str), y=top_lines.values, ax=ax)
    ax.set_title("Top 10 Lines by Number of Observations")
    ax.set_xlabel("line_id")
    ax.set_ylabel("Observations")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()

In [ ]:
# 4) Distribution of key numeric features
numeric_cols = [
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "deviation_from_baseline",
]

available_numeric = [c for c in numeric_cols if c in df.columns]

if available_numeric:
    n_cols = 2
    n_rows = int(np.ceil(len(available_numeric) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
    axes = np.array(axes).reshape(-1)

    for idx, col in enumerate(available_numeric):
        sns.histplot(df[col].dropna(), bins=40, kde=True, ax=axes[idx])
        axes[idx].set_title(f"Distribution: {col}")
        axes[idx].grid(alpha=0.2)

    for idx in range(len(available_numeric), len(axes)):
        axes[idx].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No expected numeric columns found for histogram plots.")

In [ ]:
# 5) Hourly delay rate pattern
hourly_delay = (
    df.groupby("hour", as_index=False)["late_3min"]
      .mean()
      .rename(columns={"late_3min": "late_rate"})
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=hourly_delay, x="hour", y="late_rate", marker="o", ax=ax)
ax.set_title("Delay Rate by Hour of Day")
ax.set_xlabel("Hour")
ax.set_ylabel("late_3min rate")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2. Sample Data for Testing ---
# Creating a smaller sample for testing purposes
df.sample(100000).to_parquet("sample_data.parquet")

In [ ]:
df.info()

In [ ]:
df.sample(5)

In [ ]:
target = "late_3min"

(
    df.corr(numeric_only=True)[target]
    .drop(target)
    .abs()
    .pipe(lambda x: x / x.sum() * 100)
    .sort_values()
    .plot(kind="barh", figsize=(8,6))
)

In [ ]:
counts = df.groupby("vehicle_id").size()
print(counts.describe())

In [ ]:
v = counts.index[0]

tmp = df[df["vehicle_id"] == v].sort_values("observed_at")

tmp[["observed_at", "time_to_station"]].head(20)

In [ ]:
tmp["dt"] = tmp["observed_at"].diff()

tmp["dt"].describe()

In [ ]:
tmp[["observed_at", "time_to_station"]].head(50)

In [ ]:
keys = ["vehicle_id", "destination_name", "direction"]

counts = df.groupby(keys).size()

print(counts.describe())

In [ ]:
k = counts.index[0]

tmp = df[
    (df["vehicle_id"] == k[0]) &
    (df["destination_name"] == k[1]) &
    (df["direction"] == k[2])
].sort_values("observed_at")

tmp[["observed_at", "time_to_station"]].head(20)

In [ ]:
keys = ["vehicle_id", "stop_id", "direction"]

counts = df.groupby(keys).size()

print(counts.describe())

In [ ]:
k = counts.index[0]

tmp = df[
    (df["vehicle_id"] == k[0]) &
    (df["stop_id"] == k[1]) &
    (df["direction"] == k[2])
].sort_values("observed_at")

tmp[["observed_at", "time_to_station"]].head(20)

In [ ]:
keys = ["vehicle_id", "stop_id", "direction", "destination_name"]

df = df.sort_values(keys + ["observed_at"])

In [ ]:
df["dt"] = df.groupby(keys)["observed_at"].diff().dt.total_seconds()

In [ ]:
GAP = 300

df["new_run"] = (df["dt"].isna()) | (df["dt"] > GAP)

In [ ]:
df["run_id"] = df.groupby(keys)["new_run"].cumsum()

In [ ]:
k = df[keys].iloc[0].tolist()

tmp = df[
    (df["vehicle_id"] == k[0]) &
    (df["stop_id"] == k[1]) &
    (df["direction"] == k[2]) &
    (df["destination_name"] == k[3])
].sort_values("observed_at")

tmp[["observed_at", "time_to_station", "run_id"]].head(50)

In [ ]:
run_counts = df.groupby(
    keys + ["run_id"]
).size()

print(run_counts.describe())

In [ ]:
# ----------------------------
# SETTINGS
# ----------------------------
HORIZON_SEC = 120
GROUP_KEYS = ["vehicle_id", "stop_id", "direction", "destination_name", "run_id"]

# make sure sorted correctly
df = df.sort_values(GROUP_KEYS + ["observed_at"]).reset_index(drop=True)

# store original row id so we can merge back later
df["row_id"] = np.arange(len(df))

future_parts = []

for key, g in df.groupby(GROUP_KEYS, sort=False):
    g = g.sort_values("observed_at").copy()

    # target future time for each row
    g["target_time"] = g["observed_at"] + pd.Timedelta(seconds=HORIZON_SEC)

    # prepare right-side table: future candidates within same run
    right = g[["observed_at", "late_3min"]].copy()
    right = right.rename(columns={
        "observed_at": "future_observed_at",
        "late_3min": "future_late_3min_120s"
    })

    # merge_asof: for each target_time, find first future row with time >= target_time
    matched = pd.merge_asof(
        g.sort_values("target_time"),
        right.sort_values("future_observed_at"),
        left_on="target_time",
        right_on="future_observed_at",
        direction="forward"
    )

    future_parts.append(
        matched[["row_id", "target_time", "future_observed_at", "future_late_3min_120s"]]
    )

future_df = pd.concat(future_parts, ignore_index=True)

# merge back to main dataframe
df = df.merge(future_df, on="row_id", how="left")

# optional: how many rows got a future label?
print(df["future_late_3min_120s"].isna().mean(), "fraction missing future label")

# keep only rows where future label exists
df_path_b = df.dropna(subset=["future_late_3min_120s"]).copy()

# make target integer
df_path_b["future_late_3min_120s"] = df_path_b["future_late_3min_120s"].astype(int)

print("Original rows:", len(df))
print("Path B rows with valid future target:", len(df_path_b))
print(df_path_b["future_late_3min_120s"].value_counts(dropna=False))

In [ ]:
print("Original rows:", len(df))
print("Path B rows with valid future target:", len(df_path_b))
print(df_path_b["future_late_3min_120s"].value_counts(dropna=False))

sample_cols = [
    "observed_at",
    "target_time",
    "future_observed_at",
    "late_3min",
    "future_late_3min_120s",
    "vehicle_id",
    "stop_id",
    "direction",
    "destination_name",
    "run_id"
]

print(df_path_b[sample_cols].head(20))

In [ ]:
df.sample(5)

In [ ]:
df.info()

In [ ]:
df_path_b.info()

In [ ]:
df_path_b.sample(5)

In [ ]:
print(df_path_b["observed_at"].min())
print(df_path_b["observed_at"].max())
print(df_path_b["observed_at"].describe())

In [ ]:
df_path_b = df_path_b.sort_values("observed_at")

t1 = df_path_b["observed_at"].quantile(0.7)
t2 = df_path_b["observed_at"].quantile(0.85)

print("t1 =", t1)
print("t2 =", t2)

train_df = df_path_b[df_path_b["observed_at"] < t1]
val_df   = df_path_b[(df_path_b["observed_at"] >= t1) & (df_path_b["observed_at"] < t2)]
test_df  = df_path_b[df_path_b["observed_at"] >= t2]

print("train:", len(train_df))
print("val:", len(val_df))
print("test:", len(test_df))

In [ ]:
print(df_path_b.columns.tolist())

In [ ]:
df_path_b.info(verbose=True, show_counts=True)

In [ ]:
drop_cols = [
    "future_late_3min_120s",
    "future_observed_at",
    "target_time",
    "row_id",
    "run_id",
    "new_run",
    "dt",
    "late",
    "late_3min",
    "observed_at"
]

feature_cols = [c for c in train_df.columns if c not in drop_cols]

print("Features:", feature_cols)
print("N features:", len(feature_cols))

X_train = train_df[feature_cols]
y_train = train_df["future_late_3min_120s"]

X_val = val_df[feature_cols]
y_val = val_df["future_late_3min_120s"]

X_test = test_df[feature_cols]
y_test = test_df["future_late_3min_120s"]

print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

# ----------------------------
# TARGET
# ----------------------------
target_col = "future_late_3min_120s"

# ----------------------------
# FEATURE GROUPS
# ----------------------------
categorical_features = [
    "stop_id",
    "stop_name",
    "line_id",
    "vehicle_id",
    "direction",
    "platform_name",
    "destination_name",
]

numeric_features = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

feature_cols = categorical_features + numeric_features

# ----------------------------
# X / y
# ----------------------------
X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)

In [ ]:
# ----------------------------
# PREPROCESSORS
# ----------------------------

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

numeric_transformer_lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor_rf = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", numeric_transformer_rf, numeric_features),
    ]
)

preprocessor_lr = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", numeric_transformer_lr, numeric_features),
    ]
)

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_rf),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=20,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced_subsample"
    ))
])

print("Fitting Random Forest...")
rf_pipeline.fit(X_train, y_train)

val_proba_rf = rf_pipeline.predict_proba(X_val)[:, 1]
val_pred_rf = (val_proba_rf >= 0.5).astype(int)

print("\n=== RANDOM FOREST: VALIDATION REPORT ===")
print(classification_report(y_val, val_pred_rf, digits=4))
print("ROC-AUC:", roc_auc_score(y_val, val_proba_rf))
print("PR-AUC :", average_precision_score(y_val, val_proba_rf))
print("Confusion Matrix:\n", confusion_matrix(y_val, val_pred_rf))

In [ ]:
# =========================
# MODEL V1 (already trained)
# keep these as reference
# =========================
rf_pipeline_v1 = rf_pipeline
val_proba_rf_v1 = val_proba_rf
val_pred_rf_v1 = val_pred_rf

# optional: save metrics in a dict
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

results = {}

results["rf_v1_with_vehicle_id"] = {
    "roc_auc": roc_auc_score(y_val, val_proba_rf_v1),
    "pr_auc": average_precision_score(y_val, val_proba_rf_v1),
    "report": classification_report(y_val, val_pred_rf_v1, digits=4),
    "confusion_matrix": confusion_matrix(y_val, val_pred_rf_v1),
}

print(results["rf_v1_with_vehicle_id"]["report"])
print(results["rf_v1_with_vehicle_id"]["confusion_matrix"])

In [ ]:
# =========================
# MODEL V2 (without vehicle_id)
# =========================
categorical_features_v2 = [
    "stop_id",
    "stop_name",
    "line_id",
    "direction",
    "platform_name",
    "destination_name",
]

numeric_features_v2 = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

feature_cols_v2 = categorical_features_v2 + numeric_features_v2

X_train_v2 = train_df[feature_cols_v2].copy()
X_val_v2   = val_df[feature_cols_v2].copy()
X_test_v2  = test_df[feature_cols_v2].copy()

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

categorical_transformer_v2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer_v2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor_rf_v2 = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer_v2, categorical_features_v2),
        ("num", numeric_transformer_v2, numeric_features_v2),
    ]
)

rf_pipeline_v2 = Pipeline(steps=[
    ("preprocessor", preprocessor_rf_v2),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=20,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced_subsample"
    ))
])

print("Fitting Random Forest v2 (without vehicle_id)...")
rf_pipeline_v2.fit(X_train_v2, y_train)

val_proba_rf_v2 = rf_pipeline_v2.predict_proba(X_val_v2)[:, 1]
val_pred_rf_v2 = (val_proba_rf_v2 >= 0.5).astype(int)

In [ ]:
results["rf_v2_without_vehicle_id"] = {
    "roc_auc": roc_auc_score(y_val, val_proba_rf_v2),
    "pr_auc": average_precision_score(y_val, val_proba_rf_v2),
    "report": classification_report(y_val, val_pred_rf_v2, digits=4),
    "confusion_matrix": confusion_matrix(y_val, val_pred_rf_v2),
}

print(results["rf_v2_without_vehicle_id"]["report"])
print(results["rf_v2_without_vehicle_id"]["confusion_matrix"])

In [ ]:
print("=== COMPARISON ===")
print("V1 ROC-AUC:", results["rf_v1_with_vehicle_id"]["roc_auc"])
print("V2 ROC-AUC:", results["rf_v2_without_vehicle_id"]["roc_auc"])
print("V1 PR-AUC :", results["rf_v1_with_vehicle_id"]["pr_auc"])
print("V2 PR-AUC :", results["rf_v2_without_vehicle_id"]["pr_auc"])

## Checking the same model with numerical features only

In [ ]:
numeric_features_v3 = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

# ----------------------------
# V3: numeric only
# ----------------------------
numeric_features_v3 = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

X_train_v3 = train_df[numeric_features_v3].copy()
X_val_v3   = val_df[numeric_features_v3].copy()
X_test_v3  = test_df[numeric_features_v3].copy()

numeric_pipeline_v3 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=20,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced_subsample"
    ))
])

print("Fitting Random Forest v3 (numeric only)...")
numeric_pipeline_v3.fit(X_train_v3, y_train)

val_proba_rf_v3 = numeric_pipeline_v3.predict_proba(X_val_v3)[:, 1]
val_pred_rf_v3 = (val_proba_rf_v3 >= 0.5).astype(int)

print("\n=== RANDOM FOREST V3: VALIDATION REPORT ===")
print(classification_report(y_val, val_pred_rf_v3, digits=4))
print("ROC-AUC:", roc_auc_score(y_val, val_proba_rf_v3))
print("PR-AUC :", average_precision_score(y_val, val_proba_rf_v3))
print("Confusion Matrix:\n", confusion_matrix(y_val, val_pred_rf_v3))

In [ ]:
results["rf_v3_numeric_only"] = {
    "roc_auc": roc_auc_score(y_val, val_proba_rf_v3),
    "pr_auc": average_precision_score(y_val, val_proba_rf_v3),
    "report": classification_report(y_val, val_pred_rf_v3, digits=4),
    "confusion_matrix": confusion_matrix(y_val, val_pred_rf_v3),
}

print("=== COMPARISON ===")
print("V1 ROC-AUC:", results["rf_v1_with_vehicle_id"]["roc_auc"])
print("V2 ROC-AUC:", results["rf_v2_without_vehicle_id"]["roc_auc"])
print("V3 ROC-AUC:", results["rf_v3_numeric_only"]["roc_auc"])

print("V1 PR-AUC :", results["rf_v1_with_vehicle_id"]["pr_auc"])
print("V2 PR-AUC :", results["rf_v2_without_vehicle_id"]["pr_auc"])
print("V3 PR-AUC :", results["rf_v3_numeric_only"]["pr_auc"])

# Trying with future_late_3min_300s rather than future_late_3min_120s
i.e. with 5 minutes window rather than 2 minutes

In [ ]:
import pandas as pd
import numpy as np

# ----------------------------
# SETTINGS
# ----------------------------
HORIZON_SEC = 300
GROUP_KEYS = ["vehicle_id", "stop_id", "direction", "destination_name", "run_id"]

# We use the existing df that already has run_id
df = df.sort_values(GROUP_KEYS + ["observed_at"]).reset_index(drop=True)

# If row_id already exists, keep it; otherwise create it
if "row_id" not in df.columns:
    df["row_id"] = np.arange(len(df))

future_parts_300 = []

for key, g in df.groupby(GROUP_KEYS, sort=False):
    g = g.sort_values("observed_at").copy()

    g["target_time_300s"] = g["observed_at"] + pd.Timedelta(seconds=HORIZON_SEC)

    right = g[["observed_at", "late_3min"]].copy()
    right = right.rename(columns={
        "observed_at": "future_observed_at_300s",
        "late_3min": "future_late_3min_300s"
    })

    matched = pd.merge_asof(
        g.sort_values("target_time_300s"),
        right.sort_values("future_observed_at_300s"),
        left_on="target_time_300s",
        right_on="future_observed_at_300s",
        direction="forward"
    )

    future_parts_300.append(
        matched[["row_id", "target_time_300s", "future_observed_at_300s", "future_late_3min_300s"]]
    )

future_df_300 = pd.concat(future_parts_300, ignore_index=True)

# Merge back
df_300 = df.merge(future_df_300, on="row_id", how="left")

# Keep only rows with valid future target
df_path_b_300 = df_300.dropna(subset=["future_late_3min_300s"]).copy()
df_path_b_300["future_late_3min_300s"] = df_path_b_300["future_late_3min_300s"].astype(int)

print("Original rows:", len(df))
print("300s rows with valid future target:", len(df_path_b_300))
print("Missing fraction:", df_300["future_late_3min_300s"].isna().mean())
print(df_path_b_300["future_late_3min_300s"].value_counts(dropna=False))

In [ ]:
df_path_b_300 = df_path_b_300.sort_values("observed_at")

t1_300 = df_path_b_300["observed_at"].quantile(0.7)
t2_300 = df_path_b_300["observed_at"].quantile(0.85)

train_df_300 = df_path_b_300[df_path_b_300["observed_at"] < t1_300]
val_df_300   = df_path_b_300[(df_path_b_300["observed_at"] >= t1_300) & (df_path_b_300["observed_at"] < t2_300)]
test_df_300  = df_path_b_300[df_path_b_300["observed_at"] >= t2_300]

print("t1_300 =", t1_300)
print("t2_300 =", t2_300)
print("train_300:", len(train_df_300))
print("val_300:", len(val_df_300))
print("test_300:", len(test_df_300))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

target_col_300 = "future_late_3min_300s"

numeric_features_v3 = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

X_train_300 = train_df_300[numeric_features_v3].copy()
y_train_300 = train_df_300[target_col_300].copy()

X_val_300 = val_df_300[numeric_features_v3].copy()
y_val_300 = val_df_300[target_col_300].copy()

numeric_pipeline_v3_300 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=20,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced_subsample"
    ))
])

print("Fitting Random Forest v3 (numeric only, 300s horizon)...")
numeric_pipeline_v3_300.fit(X_train_300, y_train_300)

val_proba_rf_v3_300 = numeric_pipeline_v3_300.predict_proba(X_val_300)[:, 1]
val_pred_rf_v3_300 = (val_proba_rf_v3_300 >= 0.5).astype(int)

print("\n=== RANDOM FOREST V3: VALIDATION REPORT (300s) ===")
print(classification_report(y_val_300, val_pred_rf_v3_300, digits=4))
print("ROC-AUC:", roc_auc_score(y_val_300, val_proba_rf_v3_300))
print("PR-AUC :", average_precision_score(y_val_300, val_proba_rf_v3_300))
print("Confusion Matrix:\n", confusion_matrix(y_val_300, val_pred_rf_v3_300))

## Features for 300s V2 (numeric + categorical without vehicle_id)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

target_col_300 = "future_late_3min_300s"

numeric_features = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

categorical_features = [
    "stop_id",
    "line_id",
    "direction",
    "destination_name",
    "platform_name",
]

features_v2_300 = numeric_features + categorical_features

X_train_300 = train_df_300[features_v2_300].copy()
y_train_300 = train_df_300[target_col_300].copy()

X_val_300 = val_df_300[features_v2_300].copy()
y_val_300 = val_df_300[target_col_300].copy()


numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced_subsample"
)

pipeline_v2_300 = Pipeline([
    ("prep", preprocessor),
    ("model", rf_model),
])

print("Fitting Random Forest V2 (numeric + categorical, 300s)...")
pipeline_v2_300.fit(X_train_300, y_train_300)

val_proba_v2_300 = pipeline_v2_300.predict_proba(X_val_300)[:, 1]
val_pred_v2_300 = (val_proba_v2_300 >= 0.5).astype(int)

print("\n=== RANDOM FOREST V2 (300s) ===")
print(classification_report(y_val_300, val_pred_v2_300, digits=4))
print("ROC-AUC:", roc_auc_score(y_val_300, val_proba_v2_300))
print("PR-AUC :", average_precision_score(y_val_300, val_proba_v2_300))
print("Confusion Matrix:\n", confusion_matrix(y_val_300, val_pred_v2_300))

In [ ]:
print("=== 300s COMPARISON ===")

print("Numeric only ROC:", roc_auc_score(y_val_300, val_proba_rf_v3_300))
print("Numeric+Cat ROC:", roc_auc_score(y_val_300, val_proba_v2_300))

print("Numeric only PR :", average_precision_score(y_val_300, val_proba_rf_v3_300))
print("Numeric+Cat PR :", average_precision_score(y_val_300, val_proba_v2_300))

## Another check using Horizon = 1800 seconds or 30 minutes

In [ ]:
import pandas as pd
import numpy as np

HORIZON_SEC = 1800

GROUP_KEYS = [
    "vehicle_id",
    "stop_id",
    "direction",
    "destination_name",
    "run_id",
]

df = df.sort_values(GROUP_KEYS + ["observed_at"]).reset_index(drop=True)

if "row_id" not in df.columns:
    df["row_id"] = np.arange(len(df))


future_parts_1800 = []

for key, g in df.groupby(GROUP_KEYS, sort=False):

    g = g.sort_values("observed_at").copy()

    g["target_time_1800s"] = (
        g["observed_at"] + pd.Timedelta(seconds=HORIZON_SEC)
    )

    right = g[["observed_at", "late_3min"]].copy()

    right = right.rename(
        columns={
            "observed_at": "future_observed_at_1800s",
            "late_3min": "future_late_3min_1800s",
        }
    )

    matched = pd.merge_asof(
        g.sort_values("target_time_1800s"),
        right.sort_values("future_observed_at_1800s"),
        left_on="target_time_1800s",
        right_on="future_observed_at_1800s",
        direction="forward",
    )

    future_parts_1800.append(
        matched[
            [
                "row_id",
                "target_time_1800s",
                "future_observed_at_1800s",
                "future_late_3min_1800s",
            ]
        ]
    )


future_df_1800 = pd.concat(future_parts_1800, ignore_index=True)

df_1800 = df.merge(future_df_1800, on="row_id", how="left")

df_path_b_1800 = df_1800.dropna(
    subset=["future_late_3min_1800s"]
).copy()

df_path_b_1800["future_late_3min_1800s"] = (
    df_path_b_1800["future_late_3min_1800s"].astype(int)
)

print("Original rows:", len(df))
print("Valid rows 1800s:", len(df_path_b_1800))
print(
    "Missing fraction:",
    df_1800["future_late_3min_1800s"].isna().mean(),
)

print(
    df_path_b_1800["future_late_3min_1800s"].value_counts()
)

In [ ]:
df_path_b_1800 = df_path_b_1800.sort_values("observed_at")

t1_1800 = df_path_b_1800["observed_at"].quantile(0.7)
t2_1800 = df_path_b_1800["observed_at"].quantile(0.85)

train_df_1800 = df_path_b_1800[
    df_path_b_1800["observed_at"] < t1_1800
]

val_df_1800 = df_path_b_1800[
    (df_path_b_1800["observed_at"] >= t1_1800)
    & (df_path_b_1800["observed_at"] < t2_1800)
]

test_df_1800 = df_path_b_1800[
    df_path_b_1800["observed_at"] >= t2_1800
]

print("t1:", t1_1800)
print("t2:", t2_1800)

print("train:", len(train_df_1800))
print("val:", len(val_df_1800))
print("test:", len(test_df_1800))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

target_col_1800 = "future_late_3min_1800s"

numeric_features = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

X_train_1800 = train_df_1800[numeric_features]
y_train_1800 = train_df_1800[target_col_1800]

X_val_1800 = val_df_1800[numeric_features]
y_val_1800 = val_df_1800[target_col_1800]


pipeline_1800 = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=12,
                min_samples_leaf=20,
                n_jobs=-1,
                random_state=42,
                class_weight="balanced_subsample",
            ),
        ),
    ]
)

print("Fitting RF 1800s...")
pipeline_1800.fit(X_train_1800, y_train_1800)

proba_1800 = pipeline_1800.predict_proba(X_val_1800)[:, 1]
pred_1800 = (proba_1800 >= 0.5).astype(int)

print("\n=== 1800s RESULT ===")

print(classification_report(y_val_1800, pred_1800, digits=4))

print("ROC:", roc_auc_score(y_val_1800, proba_1800))
print("PR :", average_precision_score(y_val_1800, proba_1800))

print(
    confusion_matrix(y_val_1800, pred_1800)
)

## Logistic Regression for horizon = 5 minutes

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

target_col = "future_late_3min_300s"

numeric_features = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

X_train = train_df_300[numeric_features]
y_train = train_df_300[target_col]

X_val = val_df_300[numeric_features]
y_val = val_df_300[target_col]


log_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=200,
                n_jobs=-1,
            ),
        ),
    ]
)

print("Fitting Logistic Regression...")

log_pipeline.fit(X_train, y_train)

proba_log = log_pipeline.predict_proba(X_val)[:, 1]
pred_log = (proba_log >= 0.5).astype(int)

print("\n=== LOGISTIC RESULT (300s) ===")

print(classification_report(y_val, pred_log, digits=4))

print("ROC:", roc_auc_score(y_val, proba_log))
print("PR :", average_precision_score(y_val, proba_log))

print(confusion_matrix(y_val, pred_log))

### Logistic Regression for horizon = 30 minutes

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

target_col_1800 = "future_late_3min_1800s"

numeric_features = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

X_train_1800 = train_df_1800[numeric_features].copy()
y_train_1800 = train_df_1800[target_col_1800].copy()

X_val_1800 = val_df_1800[numeric_features].copy()
y_val_1800 = val_df_1800[target_col_1800].copy()

log_pipeline_1800 = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=300,
            class_weight="balanced",
            random_state=42
        )),
    ]
)

print("Fitting Logistic Regression 1800s...")
log_pipeline_1800.fit(X_train_1800, y_train_1800)

proba_log_1800 = log_pipeline_1800.predict_proba(X_val_1800)[:, 1]
pred_log_1800 = (proba_log_1800 >= 0.5).astype(int)

print("\n=== LOGISTIC RESULT (1800s) ===")
print(classification_report(y_val_1800, pred_log_1800, digits=4))
print("ROC:", roc_auc_score(y_val_1800, proba_log_1800))
print("PR :", average_precision_score(y_val_1800, proba_log_1800))
print(confusion_matrix(y_val_1800, pred_log_1800))

## LightGBM for horizon = 5 minutes

In [ ]:
import lightgbm as lgb
print(lgb.__version__)

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

target_col_300 = "future_late_3min_300s"

numeric_features = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

X_train_300 = train_df_300[numeric_features].copy()
y_train_300 = train_df_300[target_col_300].copy()

X_val_300 = val_df_300[numeric_features].copy()
y_val_300 = val_df_300[target_col_300].copy()

lgbm_pipeline_300 = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=-1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1,
            verbose=-1
        )),
    ]
)

print("Fitting LightGBM 300s (CPU)...")
lgbm_pipeline_300.fit(X_train_300, y_train_300)

proba_lgbm_300 = lgbm_pipeline_300.predict_proba(X_val_300)[:, 1]
pred_lgbm_300 = (proba_lgbm_300 >= 0.5).astype(int)

print("\n=== LIGHTGBM RESULT (300s, CPU) ===")
print(classification_report(y_val_300, pred_lgbm_300, digits=4))
print("ROC:", roc_auc_score(y_val_300, proba_lgbm_300))
print("PR :", average_precision_score(y_val_300, proba_lgbm_300))
print(confusion_matrix(y_val_300, pred_lgbm_300))

#### LightGBM with GPU

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

lgbm_pipeline_300_gpu = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=-1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1,
            verbose=-1,
            device="gpu"
        )),
    ]
)

print("Fitting LightGBM 300s (GPU)...")
lgbm_pipeline_300_gpu.fit(X_train_300, y_train_300)

proba_lgbm_300_gpu = lgbm_pipeline_300_gpu.predict_proba(X_val_300)[:, 1]
pred_lgbm_300_gpu = (proba_lgbm_300_gpu >= 0.5).astype(int)

print("\n=== LIGHTGBM RESULT (300s, GPU) ===")
print(classification_report(y_val_300, pred_lgbm_300_gpu, digits=4))
print("ROC:", roc_auc_score(y_val_300, proba_lgbm_300_gpu))
print("PR :", average_precision_score(y_val_300, proba_lgbm_300_gpu))
print(confusion_matrix(y_val_300, pred_lgbm_300_gpu))

In [ ]:
print("=== MODEL COMPARISON ===")

print("LOG 300 ROC:", roc_auc_score(y_val_300, proba_log))
print("LOG 300 PR :", average_precision_score(y_val_300, proba_log))

print("LGBM 300 ROC (CPU):", roc_auc_score(y_val_300, proba_lgbm_300))
print("LGBM 300 PR  (CPU):", average_precision_score(y_val_300, proba_lgbm_300))

print("LGBM 300 ROC (GPU):", roc_auc_score(y_val_300, proba_lgbm_300_gpu))
print("LGBM 300 PR  (GPU):", average_precision_score(y_val_300, proba_lgbm_300_gpu))

print("LOG 1800 ROC:", roc_auc_score(y_val_1800, proba_log_1800))
print("LOG 1800 PR :", average_precision_score(y_val_1800, proba_log_1800))

In [ ]:
print("X_train_1800:", X_train_1800.shape)
print("y_train_1800:", y_train_1800.shape)
print("X_val_1800:", X_val_1800.shape)
print("y_val_1800:", y_val_1800.shape)

print("Target train distribution:")
print(y_train_1800.value_counts(dropna=False))

print("Target val distribution:")
print(y_val_1800.value_counts(dropna=False))

In [ ]:
print("LOG 1800 ROC:", roc_auc_score(y_val_1800, proba_log_1800))
print("LOG 1800 PR :", average_precision_score(y_val_1800, proba_log_1800))

In [ ]:
pred_log_1800 = (proba_log_1800 >= 0.5).astype(int)

print(classification_report(y_val_1800, pred_log_1800, digits=4))
print(confusion_matrix(y_val_1800, pred_log_1800))